In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\User\Downloads\Analysis on Music streaming platforms\Data\raw_data\raw_form_data.csv")
df.head(5)

,Timestamp,Age group,Which music app do you use the most?,How many hours do you listen to music daily?,Which device do you use the most for listening?,How often do you skip songs?,Do you watch music videos along with listening?,What describes your listening behaviour best?,Do you use Free or Premium?,Why did you choose your current app?,Overall satisfaction with the app,Recommendation quality,Is the premium price worth it?,What is your biggest frustration with your current app?,"If you could change one thing in your app, what would it be?",Have you ever switched apps before?,"If yes, why did you switch?",Would you switch again if a better app comes?
0,2025/12/12 3:40:04 pm GMT+5:30,35–44,Spotify,2–4 hours,Phone,Rarely,No,I like discovering new music,Free,Habit / long-time user,6.0,6.0,NaN,None of the above,YouTube music,Yes,Missing songs,Maybe
1,2025/12/12 3:50:36 pm GMT+5:30,18–24,Spotify,<1 hour,Phone,Rarely,Yes,I mostly listen to my saved playlists,Premium,Price,6.0,8.0,10.0,None of the above,Premium Subscription,Yes,Too many ads,No
2,2025/12/12 3:52:21 pm GMT+5:30,18–24,Spotify,1–2 hours,Phone,Often,No,Both equally,Free,Music quality;Better recommendations;Song vari...,5.0,5.0,4.0,Poor recommendations,NaN,Yes,Too many ads,Yes
3,2025/12/12 3:57:19 pm GMT+5:30,18–24,Spotify,<1 hour,Phone,Rarely,Yes,I mostly listen to my saved playlists,Free,Music quality;Better recommendations;Large per...,7.0,9.0,9.0,Too many ads,Too many adds,Yes,Too many ads,No
4,2025/12/12 3:58:27 pm GMT+5:30,18–24,Spotify,<1 hour,Phone,Sometimes,No,I mostly listen to my saved playlists,Free,Music quality;Habit / long-time user,8.0,8.0,8.0,Too many ads,NaN,No,NaN,Maybe


In [2]:
import pandas as pd
import numpy as np
import re



# Backup (optional but smart)
df_raw = df.copy()

# =========================
# 1) CLEAN COLUMN NAMES
# =========================
df.columns = (
    df.columns
    .astype(str)
    .str.strip(
        
    )
    .str.replace(r'\s+', '_', regex=True)     # any whitespace -> _
    .str.replace('?', '', regex=False)
    .str.replace(',', '', regex=False)
)

# =========================
# 2) STRIP STRING VALUES + NORMALIZE DASHES
# =========================
obj_cols = df.select_dtypes(include='object').columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip()

# Normalize common dash variants (en-dash/em-dash) used in ranges
def normalize_dashes(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace('–', '-', regex=False)
             .str.replace('—', '-', regex=False)
             .str.replace('\u2212', '-', regex=False)  # minus sign
             .str.replace(r'\s+', ' ', regex=True)
             .str.strip()
           )

# =========================
# 3) DROP DUPLICATES (safe)
# =========================
df = df.drop_duplicates()

# =========================
# 4) HANDLE NULLS / BLANKS (dashboard-safe)
# =========================
# Convert empty strings like "nan" to real NaN for controlled filling
df = df.replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# Fill small missing categorical fields (if any)
if 'How_often_do_you_skip_songs' in df.columns:
    df['How_often_do_you_skip_songs'] = df['How_often_do_you_skip_songs'].fillna('Unknown')

# =========================
# 5) ORDINAL ENCODING: LISTENING HOURS
# =========================
hours_col = 'How_many_hours_do_you_listen_to_music_daily'
if hours_col in df.columns:
    df[hours_col] = normalize_dashes(df[hours_col])

    # Make mapping flexible: handles "<1 hour", "1-2 hours", "2-4 hours", "4+ hours"
    hours_map = {
        '<1 hour': 1,
        '1-2 hours': 2,
        '2-4 hours': 3,
        '4+ hours': 4,
        'unknown': 0
    }

    # Lowercase for stable matching
    hours_norm = df[hours_col].fillna('unknown').astype(str).str.lower()

    # Standardize a few common variants
    hours_norm = (hours_norm
                  .str.replace('less than 1 hour', '<1 hour', regex=False)
                  .str.replace('1 - 2 hours', '1-2 hours', regex=False)
                  .str.replace('2 - 4 hours', '2-4 hours', regex=False)
                  .str.replace('4 + hours', '4+ hours', regex=False)
                 )

    df['listening_hours_ord'] = hours_norm.map(hours_map)

# =========================
# 6) ORDINAL ENCODING: SKIP FREQUENCY (optional; only if values are words)
# =========================
skip_col = 'How_often_do_you_skip_songs'
if skip_col in df.columns:
    skip_norm = df[skip_col].fillna('Unknown').astype(str).str.strip().str.lower()

    # If your values are already numeric, this map won't apply; it will produce NaN.
    # That's okay; we handle it by trying numeric conversion too.
    skip_map = {
        'never': 1,
        'rarely': 2,
        'sometimes': 3,
        'often': 4,
        'very often': 5,
        'always': 5,
        'unknown': 0
    }

    df['skip_frequency_ord'] = skip_norm.map(skip_map)

    # If mapping failed (many NaNs), attempt numeric coercion instead
    if df['skip_frequency_ord'].isna().mean() > 0.5:
        df['skip_frequency_ord'] = pd.to_numeric(df[skip_col], errors='coerce').fillna(0).astype(int)

# =========================
# 7) NUMERIC CLEANING: SATISFACTION / RECOMMENDATION / PREMIUM WORTH IT
# =========================
num_cols = [
    'Overall_satisfaction_with_the_app',
    'Recommendation_quality',
    'Is_the_premium_price_worth_it'
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Fill satisfaction + recommendation with median (safe default)
for c in ['Overall_satisfaction_with_the_app', 'Recommendation_quality']:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())

# =========================
# 8) BINARY FLAGS (dashboard KPIs)
# =========================
# Premium user
prem_col = 'Do_you_use_Free_or_Premium'
if prem_col in df.columns:
    prem_norm = df[prem_col].fillna('Unknown').astype(str).str.strip().str.lower()
    df['premium_user'] = prem_norm.map({'free': 0, 'premium': 1}).fillna(0).astype(int)

# Watches videos
vid_col = 'Do_you_watch_music_videos_along_with_listening'
if vid_col in df.columns:
    vid_norm = df[vid_col].fillna('Unknown').astype(str).str.strip().str.lower()
    df['watches_video'] = vid_norm.map({'yes': 1, 'no': 0}).fillna(0).astype(int)

# Switched before
sw_col = 'Have_you_ever_switched_apps_before'
if sw_col in df.columns:
    sw_norm = df[sw_col].fillna('Unknown').astype(str).str.strip().str.lower()
    df['switched_before'] = sw_norm.map({'yes': 1, 'no': 0}).fillna(0).astype(int)

# Switch again risk
risk_col = 'Would_you_switch_again_if_a_better_app_comes'
if risk_col in df.columns:
    risk_norm = df[risk_col].fillna('Unknown').astype(str).str.strip().str.lower()
    df['switch_risk'] = risk_norm.map({'yes': 1, 'no': 0}).fillna(0).astype(int)

# =========================
# 9) CONDITIONAL: "Premium price worth it" should apply mainly to premium users
# =========================
worth_col = 'Is_the_premium_price_worth_it'
if worth_col in df.columns and 'premium_user' in df.columns:
    # Dashboard-safe column: score for premium users, 0 for non-premium / not-applicable
    df['premium_price_worth_it_score'] = df[worth_col].where(df['premium_user'] == 1, 0)
    df['premium_price_applicable'] = (df['premium_user'] == 1).astype(int)
else:
    if worth_col in df.columns:
        df['premium_price_worth_it_score'] = df[worth_col].fillna(0)
        df['premium_price_applicable'] = (df[worth_col].notna()).astype(int)

# =========================
# 10) MULTI-SELECT: "Why did you choose your current app?" -> dummy columns
# =========================
choose_col = 'Why_did_you_choose_your_current_app'
if choose_col in df.columns:
    # Ensure string & normalize separators
    df[choose_col] = df[choose_col].fillna('').astype(str).str.strip()
    df[choose_col] = normalize_dashes(df[choose_col])  # just in case
    # Common separator is ';' (if commas exist, convert to ';')
    df[choose_col] = df[choose_col].str.replace(', ', ';', regex=False)

    dummies = df[choose_col].str.get_dummies(sep=';')
    dummies.columns = (
        dummies.columns
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', '_', regex=True)
        .str.replace('+', '', regex=False)
        .str.replace('/', '', regex=False)
        .str.replace('-', '_', regex=False)
    )

    # Remove empty column if present
    if '' in dummies.columns:
        dummies = dummies.drop(columns=[''])

    df = pd.concat([df, dummies], axis=1)

    # Drop raw multi-select column to keep dashboard clean
    df = df.drop(columns=[choose_col])

# =========================
# 11) CLEAN UGLY COLUMN NAMES (double underscores etc.)
# =========================
df.columns = (pd.Index(df.columns)
              .str.replace(r'__+', '_', regex=True)
              .str.strip('_')
             )

# =========================
# 12) DASHBOARD-FRIENDLY ENGINEERED METRICS
# =========================
# Satisfaction tier (business readable)
if 'Overall_satisfaction_with_the_app' in df.columns:
    df['satisfaction_tier'] = pd.cut(
        df['Overall_satisfaction_with_the_app'],
        bins=[-np.inf, 5, 7, np.inf],
        labels=['Low', 'Medium', 'High']
    )

# Engagement score (simple but powerful)
if 'listening_hours_ord' in df.columns and 'skip_frequency_ord' in df.columns:
    # Lower skipping is better; convert skip to a positive score (optional logic)
    # If skip_frequency_ord is 1 (never) -> 4 points; if 5 -> 0 points
    df['skip_score'] = (6 - df['skip_frequency_ord']).clip(lower=0)
    df['engagement_score'] = (df['listening_hours_ord'].fillna(0) * 2) + df['skip_score'].fillna(0)

# =========================
# 13) SET CORE CATEGORICAL DTYPES (optional but pro)
# =========================
cat_candidates = [
    'Age_group',
    'Which_music_app_do_you_use_the_most',
    'Which_device_do_you_use_the_most_for_listening',
    'What_describes_your_listening_behaviour_best',
    'How_often_do_you_skip_songs'
]
for c in cat_candidates:
    if c in df.columns:
        df[c] = df[c].astype('category')

# =========================
# 14) FINAL SANITY CHECK
# =========================
# Key columns should not be null now (except truly optional text columns you removed earlier)
check_cols = [c for c in [
    'listening_hours_ord',
    'Overall_satisfaction_with_the_app',
    'Recommendation_quality',
    'premium_price_worth_it_score'
] if c in df.columns]

print("Nulls in key engineered columns:")
print(df[check_cols].isna().sum())

# =========================
# 15) SAVE OUTPUTS
# =========================
df.to_csv('survey_data_cleaned_new.csv', index=False)

df.to_excel('survey_data_cleaned_new.xlsx', index=False)

Nulls in key engineered columns:
listening_hours_ord                  2
Overall_satisfaction_with_the_app    0
Recommendation_quality               0
premium_price_worth_it_score         0
dtype: int64
